# Task 1: Predict Restaurant Ratings (Regression)

This notebook includes scaling, target scaling, reverse scaling with `inverse_transform()`, polynomial regression, SVR, tree models, evaluation metrics, model comparison, actual-vs-predicted plots, and Random Forest feature importance.

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 2. Load Dataset

In [2]:
df = pd.read_csv("D:\B.tech\sem_5\ML\ML_internship_project\Dataset .csv")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\B.tech\\sem_5\\ML\\ML_internship_project\\Dataset .csv'

## 3. Explore Dataset

In [ ]:
print("Shape:", df.shape)
df.info()
print("\nMissing values:")
print(df.isnull().sum())
print("\nDuplicates:", df.duplicated().sum())

Shape: (9551, 21)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9551 entries, 0 to 9550
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Restaurant ID         9551 non-null   int64  
 1   Restaurant Name       9551 non-null   object 
 2   Country Code          9551 non-null   int64  
 3   City                  9551 non-null   object 
 4   Address               9551 non-null   object 
 5   Locality              9551 non-null   object 
 6   Locality Verbose      9551 non-null   object 
 7   Longitude             9551 non-null   float64
 8   Latitude              9551 non-null   float64
 9   Cuisines              9542 non-null   object 
 10  Average Cost for two  9551 non-null   int64  
 11  Currency              9551 non-null   object 
 12  Has Table booking     9551 non-null   object 
 13  Has Online delivery   9551 non-null   object 
 14  Is delivering now     9551 non-null   object 
 15  Swi

## 4. Clean Dataset

In [ ]:
df = df.drop_duplicates().copy()
df = df.dropna(subset=["Aggregate rating"]).copy()

for col in df.select_dtypes(include="object").columns:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])

for col in df.select_dtypes(include=np.number).columns:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

print("Shape after cleaning:", df.shape)

Shape after cleaning: (9551, 21)


## 5. Feature Selection

In [ ]:
target = "Aggregate rating"
drop_columns = [c for c in ["Aggregate rating","Restaurant ID","Restaurant Name","Rating text"] if c in df.columns]
X = df.drop(columns=drop_columns)
y = df[target]

print(X.columns.tolist())

['Country Code', 'City', 'Address', 'Locality', 'Locality Verbose', 'Longitude', 'Latitude', 'Cuisines', 'Average Cost for two', 'Currency', 'Has Table booking', 'Has Online delivery', 'Is delivering now', 'Switch to order menu', 'Price range', 'Rating color', 'Votes']


## 6. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

## 7. Identify Numerical and Categorical Features

In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(include="object").columns.tolist()

print("Numerical:", numeric_features)
print("Categorical:", categorical_features)

Numerical: ['Country Code', 'Longitude', 'Latitude', 'Average Cost for two', 'Price range', 'Votes']
Categorical: ['City', 'Address', 'Locality', 'Locality Verbose', 'Cuisines', 'Currency', 'Has Table booking', 'Has Online delivery', 'Is delivering now', 'Switch to order menu', 'Rating color']


## 8. StandardScaler for Feature Scaling

In [ ]:
preprocessor_scaled = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

## 9. Target Scaling

In [ ]:
y_scaler = StandardScaler()

y_train_scaled = y_scaler.fit_transform(y_train.to_numpy().reshape(-1,1)).ravel()
y_test_scaled = y_scaler.transform(y_test.to_numpy().reshape(-1,1)).ravel()

print("Original rating range:", y.min(), "to", y.max())
print("Scaled training mean:", y_train_scaled.mean())

Original rating range: 0.0 to 4.9
Scaled training mean: -5.301169625435303e-17


## 10. Evaluation Function with Reverse Scaling

In [ ]:
def evaluate_model(name, model):
    model.fit(X_train, y_train_scaled)
    pred_scaled = model.predict(X_test)

    # Reverse scaling: convert predictions back to original rating scale
    pred = y_scaler.inverse_transform(
        np.asarray(pred_scaled).reshape(-1,1)
    ).ravel()

    mse = mean_squared_error(y_test, pred)
    return {
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(y_test, pred)
    }, pred

## 11. Linear Regression

In [ ]:
linear_model = Pipeline([
    ("preprocessor", preprocessor_scaled),
    ("model", LinearRegression())
])
linear_result, linear_pred = evaluate_model("Linear Regression", linear_model)
linear_result

{'Model': 'Linear Regression',
 'MAE': 0.1626158151415699,
 'MSE': 0.0748369655276899,
 'RMSE': np.float64(0.2735634579538903),
 'R2': 0.9671206581730893}

## 12. Polynomial Regression

In [ ]:
preprocessor_poly = ColumnTransformer([
    ("num", Pipeline([
        ("scaler", StandardScaler()),
        ("poly", PolynomialFeatures(degree=2, include_bias=False))
    ]), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

polynomial_model = Pipeline([
    ("preprocessor", preprocessor_poly),
    ("model", LinearRegression())
])

poly_result, poly_pred = evaluate_model(
    "Polynomial Regression (Degree 2)", polynomial_model
)
poly_result

{'Model': 'Polynomial Regression (Degree 2)',
 'MAE': 0.1824943897048468,
 'MSE': 0.7784877326308691,
 'RMSE': np.float64(0.8823195184460497),
 'R2': 0.6579743167197729}

## 13. Support Vector Regression (SVR)

In [ ]:
svr_model = Pipeline([
    ("preprocessor", preprocessor_scaled),
    ("model", SVR(kernel="rbf", C=10, epsilon=0.1))
])

svr_result, svr_pred = evaluate_model("SVR", svr_model)
svr_result

{'Model': 'SVR',
 'MAE': 0.1800144724888307,
 'MSE': 0.05397439849187131,
 'RMSE': np.float64(0.23232390856705065),
 'R2': 0.9762865492286763}

## 14. Decision Tree Regression

In [ ]:
preprocessor_unscaled = ColumnTransformer([
    ("num", "passthrough", numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

tree_model = Pipeline([
    ("preprocessor", preprocessor_unscaled),
    ("model", DecisionTreeRegressor(max_depth=10, random_state=42))
])

tree_result, tree_pred = evaluate_model("Decision Tree Regressor", tree_model)
tree_result

{'Model': 'Decision Tree Regressor',
 'MAE': 0.12289108167901693,
 'MSE': 0.03208023815723664,
 'RMSE': np.float64(0.17910957025585383),
 'R2': 0.9859056669545184}

## 15. Random Forest Regression

In [ ]:
rf_model = Pipeline([
    ("preprocessor", preprocessor_unscaled),
    ("model", RandomForestRegressor(
        n_estimators=200, random_state=42, n_jobs=-1
    ))
])

rf_result, rf_pred = evaluate_model("Random Forest Regressor", rf_model)
rf_result

{'Model': 'Random Forest Regressor',
 'MAE': 0.11367006802722174,
 'MSE': 0.030223445185766598,
 'RMSE': np.float64(0.17384891482481735),
 'R2': 0.9867214420247701}

## 16. Model Comparison Table

In [ ]:
results_df = pd.DataFrame([
    linear_result, poly_result, svr_result, tree_result, rf_result
]).sort_values("R2", ascending=False).reset_index(drop=True)

results_df

## 17. Actual vs Predicted Plots

In [ ]:
predictions = {
    "Linear Regression": linear_pred,
    "Polynomial Regression": poly_pred,
    "SVR": svr_pred,
    "Decision Tree": tree_pred,
    "Random Forest": rf_pred
}

for name, pred in predictions.items():
    plt.figure(figsize=(7,5))
    plt.scatter(y_test, pred, alpha=0.5)
    plt.plot(
        [y_test.min(), y_test.max()],
        [y_test.min(), y_test.max()],
        linestyle="--"
    )
    plt.xlabel("Actual Rating")
    plt.ylabel("Predicted Rating")
    plt.title(f"Actual vs Predicted - {name}")
    plt.show()

## 18. Feature Importance – Random Forest

In [ ]:
rf_model.fit(X_train, y_train_scaled)

pre = rf_model.named_steps["preprocessor"]
rf = rf_model.named_steps["model"]

feature_names = pre.get_feature_names_out()
importance = pd.Series(
    rf.feature_importances_, index=feature_names
).sort_values(ascending=False)

plt.figure(figsize=(10,7))
importance.head(15).sort_values().plot(kind="barh")
plt.xlabel("Importance")
plt.title("Top 15 Random Forest Feature Importances")
plt.show()

importance.head(15)

## 19. Reverse Scaling Demonstration

In [ ]:
sample_scaled = np.array([0.0, 0.5, -0.5]).reshape(-1,1)
sample_original = y_scaler.inverse_transform(sample_scaled)

print("Scaled:", sample_scaled.ravel())
print("Original rating scale:", sample_original.ravel())

## 20. Final Conclusion

In [ ]:
best_model = results_df.iloc[0]
print("Best model:", best_model["Model"])
print("MAE:", best_model["MAE"])
print("MSE:", best_model["MSE"])
print("RMSE:", best_model["RMSE"])
print("R2:", best_model["R2"])